In [ ]:
# =====================================
# 1. IMPORT LIBRARY
# =====================================
import pandas as pd

from sqlalchemy import create_engine
from google.colab import userdata

from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

: 

In [ ]:
# =====================================
# 2. KONEKSI DATABASE
# =====================================
DB_USER = userdata.get('SUPABASE_USER')
DB_PASSWORD = userdata.get('SUPABASE_PASSWORD')
DB_HOST = userdata.get('SUPABASE_HOST')
DB_NAME = userdata.get('SUPABASE_DB')

engine = create_engine(
    f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:5432/{DB_NAME}"
)

print("✅ Koneksi database berhasil")

In [ ]:
# =====================================
# 3. QUERY DATA + SKILL
# =====================================
query = """
SELECT
  j.job_id,
  s.skill_name
FROM vw_dashboard_looker_v2 j
LEFT JOIN bridge_skill s
  ON j.job_id = s.job_id
WHERE j.gaji_midpoint IS NOT NULL
"""

df = pd.read_sql(query, engine)

print("Jumlah data awal:", len(df))

In [ ]:
# =====================================
# 4. CLEANING SKILL
# =====================================
df = df[df['skill_name'].notna()]
df = df[df['skill_name'].str.lower() != 'unknown']

In [ ]:
# =====================================
# 5. FILTER SKILL (DISKRIMINATIF)
# =====================================
skill_counts = df.groupby('skill_name')['job_id'].nunique()

filtered_skills = skill_counts[
    (skill_counts > 50) & (skill_counts < 400)
].index

df = df[df['skill_name'].isin(filtered_skills)]

print("Jumlah skill digunakan:", len(filtered_skills))

In [ ]:
# =====================================
# 6. PIVOT SKILL → MATRIX
# =====================================
df_skill = pd.crosstab(df['job_id'], df['skill_name'])

print("Shape matrix skill:", df_skill.shape)

In [ ]:
# =====================================
# 7. FEATURE (HANYA SKILL)
# =====================================
X = df_skill.copy()

In [ ]:
# # =====================================
# # 8. DROP SKILL TERLALU UMUM (OPSIONAL)
# # =====================================
drop_skills = ['SQL', 'Git']

for col in drop_skills:
    if col in X.columns:
        X = X.drop(columns=[col])

print("Shape setelah drop skill umum:", X.shape)

In [ ]:
# =====================================
# 9. 🔥 FIX: HAPUS ZERO VECTOR (WAJIB)
# =====================================
before = X.shape[0]

X = X[X.sum(axis=1) > 0]

after = X.shape[0]

print(f"Data sebelum zero-filter: {before}")
print(f"Data setelah zero-filter: {after}")

In [ ]:
# =====================================
# 10. CLUSTERING (COSINE DISTANCE)
# =====================================
agglo = AgglomerativeClustering(
    n_clusters=3,
    metric='cosine',
    linkage='average'
)

labels = agglo.fit_predict(X)

In [ ]:
# =====================================
# 11. EVALUASI
# =====================================
score = silhouette_score(X, labels, metric='cosine')

print("\n🔥 Silhouette Score:", score)

In [ ]:
# =====================================
# 12. HASIL CLUSTER
# =====================================
df_result = X.copy()
df_result['cluster'] = labels
df_result['job_id'] = df_result.index

df_result = df_result[['job_id', 'cluster']]

print("\nDistribusi cluster:")
print(df_result['cluster'].value_counts())

In [ ]:
# =====================================
# 13. VISUALISASI (PCA)
# =====================================
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels)
plt.title("Cluster Visualization (Skill-based)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

In [ ]:
# =====================================
# 14. ANALISIS CLUSTER (TOP SKILL)
# =====================================
df_analysis = X.copy()
df_analysis['cluster'] = labels

cluster_summary = df_analysis.groupby('cluster').mean()

print("\n=== TOP SKILL PER CLUSTER ===")
for c in cluster_summary.index:
    print(f"\nCluster {c}:")
    print(cluster_summary.loc[c].sort_values(ascending=False).head(10))

In [ ]:
# =====================================
# 15. SIMPAN KE DATABASE
# =====================================
df_result.to_sql(
    'job_clusters',
    engine,
    if_exists='replace',
    index=False
)

print("\n✅ Selesai! Clustering disimpan ke job_clusters")